# Phase 2: Full Data Quality Audit

This notebook performs the second major stage of the fraud detection project. The purpose of this phase is to evaluate the structural quality and analytical readiness of the full dataset before moving into deeper exploratory analysis, feature engineering, anomaly detection, or predictive modeling.

In Phase 1, a smaller sample of the dataset was used to confirm that the data could be loaded, that the expected columns were present, and that the overall structure matched the project requirements. That first step was intentionally limited because it focused on safe initial validation rather than full-scale analysis.

In this notebook, the analysis expands from a small sample to the complete dataset of more than 6.3 million rows. This allows us to answer more serious questions about data quality and fraud behavior at full scale.

The key objectives of this notebook are:

- confirm that the full dataset can be accessed and processed correctly
- verify that the data types remain appropriate at full scale
- check whether any columns contain missing values
- determine whether exact duplicate rows exist
- inspect the numeric ranges of important fields
- measure the level of class imbalance in the fraud label
- examine fraud concentration by transaction type
- create rule-based validation flags for potentially suspicious balance patterns
- isolate suspicious records for closer review
- generate and save audit outputs for later project phases

This notebook is not yet the stage where final business conclusions are made. Instead, it acts as a control and validation layer. The goal is to establish trust in the dataset, identify limitations, and highlight the patterns that deserve deeper investigation in the next phase.

## 1. Preparing the notebook environment

This first step prepares the notebook so that it can reliably access the project files and reusable Python code stored outside the notebook folder.

The notebook is being run from inside the `notebooks` directory, while some important project resources such as the `src` folder and the `data` folder are stored at the project root level. Because of that, the notebook needs to detect the correct project root and add it to Python's import search path.

This step performs four important tasks:

1. imports standard libraries that will be used in the notebook
2. identifies the correct project root directory
3. adds that project root to the Python path so that the `src` module can be imported
4. prints a small diagnostic check to confirm that the environment has been set up correctly

This is an important setup step because the rest of the notebook depends on being able to locate both the dataset and the reusable preprocessing functions.

In [40]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Using project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Using project root: /Users/twon/Documents/Time Series Analysis Project
src exists: True


## 2. Importing the reusable audit and preprocessing functions

This notebook does not define all logic directly inside the notebook. Instead, it imports reusable functions from the `src/preprocessing` module.

This is a deliberate design choice. Keeping the core audit logic inside a dedicated Python file makes the project cleaner, more modular, and easier to maintain. It also reduces duplication because the same functions can be reused in multiple notebooks or scripts later in the project.

The imported functions support several types of analysis, including:

- loading the full dataset
- counting total rows
- checking missing values in a memory-safe way
- identifying exact duplicate rows
- computing rule-based validation flags
- summarizing fraud class balance
- summarizing fraud risk by transaction type
- profiling numeric ranges
- running a complete chunked audit
- saving audit outputs to disk

At the end of this cell, the notebook also configures how pandas will display tables. This improves readability by increasing the visible number of rows and columns and formatting floating-point values in a cleaner way.

In [41]:
from src.preprocessing import (
    load_full_dataset,
    get_row_count,
    count_missing_values_chunked,
    count_exact_duplicate_rows_chunked,
    compute_validation_flags,
    summarize_validation_flags,
    summarize_flags_by_type_and_fraud,
    get_class_balance,
    get_type_risk_summary,
    profile_numeric_ranges,
    run_chunked_quality_audit,
    save_audit_outputs,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 3. Defining the dataset location and output directory

This step defines the main file paths that will be used throughout the notebook.

Two paths are important here:

- `DATA_PATH`, which points to the raw PaySim dataset file
- `OUTPUT_DIR`, which points to the folder where audit outputs will later be saved

Defining file paths near the beginning of the notebook improves readability and makes the notebook easier to adapt later. If the dataset name changes or the output folder needs to be moved, the change can be made in one place instead of throughout the notebook.

Although the path is first defined using a relative form here, the notebook will later refine the file handling approach by building paths from the project root more explicitly. That second version is more robust and better suited for a structured project.

In [42]:
DATA_PATH = Path("data/raw/paysim dataset.csv")
OUTPUT_DIR = Path("data/outputs/phase_2_audit")

## 4. Locating the raw data folder and confirming available files

Before using the dataset, it is good practice to verify that the raw data folder exists and that the expected file is present.

This step constructs the raw-data directory and output directory from the project root. It then lists all files inside the raw-data folder. The purpose is to confirm that the notebook is pointing to the correct location and that the dataset file is available under the expected name.

This is a simple but important validation step. It helps prevent path-related errors and makes the notebook behavior easier to understand for anyone reading or rerunning the project in the future.

In [43]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs" / "phase_2_audit"

print("Files in raw:")
for f in RAW_DIR.iterdir():
    print("-", f.name)

Files in raw:
- paysim dataset.csv


## 5. Selecting the dataset file and confirming that it exists

Now that the notebook has verified the contents of the raw-data folder, this step explicitly selects the PaySim dataset file and checks whether the file exists.

This matters because many later operations depend on this path being correct. If the file path is wrong, every downstream step will fail. By confirming the file's existence here, the notebook creates a clear checkpoint before moving into full-dataset processing.

This is also a useful step for project reproducibility. It makes the dataset being used explicit rather than leaving that detail implied.

In [44]:
DATA_PATH = RAW_DIR / "paysim dataset.csv"
print("file exists:", DATA_PATH.exists())

file exists: True


## 6. Counting the total number of rows in the full dataset

This step counts the total number of rows in the full dataset without loading the entire file into memory.

That is important because it gives a direct measurement of dataset size while avoiding unnecessary memory use. For a large dataset such as this one, it is good practice to separate simple structural checks from more expensive loading operations.

The output of this step helps answer a basic but important question: how large is the real working dataset that the project must handle?

Knowing the full row count also helps provide context for later calculations such as fraud rate, duplicate checks, and validation-flag frequencies.

In [45]:
total_rows = get_row_count(DATA_PATH)
print("Full dataset row count:", total_rows)

Full dataset row count: 6362620


## 7. Loading the full dataset into memory

After confirming the file path and full row count, this step loads the full dataset into a pandas DataFrame using the optimized data types defined earlier in the project.

This is one of the most important transitions in the notebook because it moves the analysis from a lightweight setup phase into full-scale dataset inspection.

The code then prints the shape of the DataFrame and displays the first few rows. These outputs help confirm three things:

- the dataset loaded successfully
- the number of rows and columns matches expectations
- the column contents look consistent with the transaction-level fraud dataset described earlier

Displaying the first few rows is especially useful because it gives a human-readable preview of the raw records and helps connect the abstract column names to actual transaction examples.

In [46]:
df_full = load_full_dataset(DATA_PATH)
print(df_full.shape)
df_full.head()

(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,"9,839.64",C1231006815,"170,136.00","160,296.36",M1979787155,0.00,0.00,0,0
1,1,PAYMENT,"1,864.28",C1666544295,"21,249.00","19,384.72",M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,0
4,1,PAYMENT,"11,668.14",C2048537720,"41,554.00","29,885.86",M1230701703,0.00,0.00,0,0


## 8. Inspecting full-dataset structure and memory usage

This step uses `df_full.info()` to display the structural summary of the full dataset.

The purpose of this inspection is to confirm:

- the number of entries in the dataset
- the names of all columns
- the assigned data type for each column
- the approximate memory usage of the loaded DataFrame

This is a key validation step because a dataset that loads successfully can still have incorrect data types or inefficient storage. Reviewing the structure at full scale helps confirm that the earlier optimization strategy is still working and that the full dataset is manageable in memory.

This output also helps document the dataset in a way that is useful for later readers of the notebook who may not be familiar with pandas.

In [47]:
df_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype   
---  ------          -----   
 0   step            int32   
 1   type            category
 2   amount          float32 
 3   nameOrig        string  
 4   oldbalanceOrg   float32 
 5   newbalanceOrig  float32 
 6   nameDest        string  
 7   oldbalanceDest  float32 
 8   newbalanceDest  float32 
 9   isFraud         int8    
 10  isFlaggedFraud  int8    
dtypes: category(1), float32(5), int32(1), int8(2), string(2)
memory usage: 388.1 MB


## 9. Checking missing values across the full dataset

This step checks whether any column in the full dataset contains missing values.

Instead of relying only on the already loaded DataFrame, the notebook performs this check through a chunked function. This means the file is scanned in smaller pieces, which is a more memory-conscious approach and also demonstrates a scalable auditing pattern.

The purpose of this step is to answer a fundamental data-quality question: is the dataset structurally complete, or are some fields missing values that could affect later analysis?

If missing values are present, they can distort descriptive statistics, break feature engineering logic, and reduce model reliability. If no missing values are found, that is also a meaningful result because it simplifies preprocessing decisions later in the project.

In [48]:
missing_values_full = count_missing_values_chunked(DATA_PATH, chunksize=250_000)
missing_values_full

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

## 10. Checking for exact duplicate rows in the full dataset

This step checks whether the full dataset contains any exact duplicate rows.

Duplicate rows matter because they can artificially inflate transaction counts, distort fraud rates, bias statistical summaries, and create misleading patterns during model training. In a transaction dataset, duplicates should generally be treated with caution unless there is a clear business reason for repeated identical records.

The duplicate check is performed in a chunked, memory-aware way using hashed row comparisons. This allows the notebook to inspect the entire dataset without requiring a second full-memory copy of the data.

The output of this step provides a direct count of exact duplicates. A zero result is usually a strong sign of structural cleanliness.

In [49]:
duplicate_rows_full = count_exact_duplicate_rows_chunked(DATA_PATH, chunksize=250_000)
print("Exact duplicate rows in full dataset:", duplicate_rows_full)

Exact duplicate rows in full dataset: 0


## 11. Profiling the numeric ranges of key variables

This step builds a compact summary of the main numeric columns in the dataset.

For each numeric field, the notebook reports values such as:

- minimum
- maximum
- mean
- median
- data type

The purpose of this profile is to create a first-pass understanding of scale, skew, and possible extreme values.

This matters because large transaction datasets often contain highly uneven distributions. For example, transaction amounts may be heavily skewed, balance fields may contain many zeros, and fraud labels are usually extremely imbalanced. Reviewing the numeric profile helps identify unusual ranges and prepares the ground for later exploratory analysis.

At this stage, the notebook is not yet deciding that any large value is necessarily an error. It is simply documenting the scale and structure of the main numeric fields.

In [50]:
numeric_profile = profile_numeric_ranges(df_full)
numeric_profile

,column,dtype,min,max,mean,median
0,step,int32,1.00,743.00,243.40,239.00
1,amount,float32,0.00,"92,445,520.00","179,861.89","74,871.94"
2,oldbalanceOrg,float32,0.00,"59,585,040.00","833,883.12","14,208.00"
3,newbalanceOrig,float32,0.00,"49,585,040.00","855,113.62",0.00
4,oldbalanceDest,float32,0.00,"356,015,904.00","1,100,701.75","132,705.66"
5,newbalanceDest,float32,0.00,"356,179,264.00","1,224,996.38","214,661.44"
6,isFraud,int8,0.00,1.00,0.00,0.00
7,isFlaggedFraud,int8,0.00,1.00,0.00,0.00


## 12. Measuring the fraud class balance

This step summarizes the distribution of the target variable `isFraud`.

The purpose is to determine how many transactions are labeled as fraud and how many are labeled as non-fraud. This is one of the most important checkpoints in the notebook because fraud detection problems are usually highly imbalanced, meaning fraudulent transactions make up only a tiny fraction of the dataset.

Understanding class balance matters for several reasons:

- it shapes how model performance should later be evaluated
- it affects how sampling or class-weighting strategies may be handled
- it provides immediate insight into how rare the fraud events are in the data

This table is one of the clearest indicators of the difficulty of the fraud detection problem.

In [51]:
class_balance = get_class_balance(df_full)
class_balance

,isFraud,count,rate
0,0,6354407,1.00
1,1,8213,0.00


## 13. Measuring fraud risk by transaction type

This step summarizes transaction activity by the `type` column.

For each transaction type, the notebook calculates:

- the total number of transactions
- the number of fraudulent transactions
- the average transaction amount
- the median transaction amount
- the fraud rate

This is a very important business-facing audit step because it begins to answer a core question of the project: which kinds of transactions appear most exposed to fraud?

Looking at fraud risk by type can reveal concentration patterns that would not be obvious from the overall dataset alone. A transaction type with a smaller volume may still be much riskier if its fraud rate is much higher than the others.

This table will become one of the foundation pieces for later exploratory analysis and feature engineering.

In [52]:
type_risk_summary = get_type_risk_summary(df_full)
type_risk_summary

,type,txn_count,fraud_count,avg_amount,median_amount,fraud_rate
4,TRANSFER,532909,4097,"910,647.00","486,308.38",0.01
1,CASH_OUT,2237500,4116,"176,273.95","147,072.19",0.00
0,CASH_IN,1399284,0,"168,920.25","143,427.71",0.00
2,DEBIT,41432,0,"5,483.67","3,048.99",0.00
3,PAYMENT,2151495,0,"13,057.60","9,482.19",0.00


## 14. Creating rule-based validation flags for the full dataset

This step applies a set of rule-based validation checks to every record in the dataset.

The result is a new DataFrame called `df_flagged` that contains all of the original transaction columns plus additional boolean flag columns. These flags are designed to highlight potentially suspicious or unusual patterns such as:

- negative transaction amounts
- negative balance values
- balance mismatches between old and new account states
- zero-balance edge cases before nonzero transactions
- merchant-like destination accounts

It is important to understand the purpose of these flags correctly. They are not final proof that a row is invalid or fraudulent. Instead, they act as screening tools that help identify records worth examining more closely.

This kind of rule-based audit is useful because it brings business logic into the data-quality process. It helps bridge the gap between raw data inspection and real fraud-focused reasoning.

In [53]:
df_flagged = compute_validation_flags(df_full)
df_flagged.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,flag_amount_negative,flag_amount_zero,flag_oldbalanceOrg_negative,flag_newbalanceOrig_negative,flag_oldbalanceDest_negative,flag_newbalanceDest_negative,sender_balance_error,flag_sender_balance_mismatch,dest_balance_error,flag_dest_balance_mismatch,flag_org_balance_zero_before_nonzero_amount,flag_dest_balance_zero_before_nonzero_amount,flag_dest_is_merchant
0,1,PAYMENT,"9,839.64",C1231006815,"170,136.00","160,296.36",M1979787155,0.00,0.00,0,0,False,False,False,False,False,False,0.00,False,"9,839.64",True,False,True,True
1,1,PAYMENT,"1,864.28",C1666544295,"21,249.00","19,384.72",M2044282225,0.00,0.00,0,0,False,False,False,False,False,False,0.00,False,"1,864.28",True,False,True,True
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0,False,False,False,False,False,False,0.00,False,181.00,True,False,True,False
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,0,False,False,False,False,False,False,0.00,False,"21,363.00",True,False,False,False
4,1,PAYMENT,"11,668.14",C2048537720,"41,554.00","29,885.86",M1230701703,0.00,0.00,0,0,False,False,False,False,False,False,0.00,False,"11,668.14",True,False,True,True


## 15. Summarizing how often each validation flag occurs

After creating the validation flags, this step measures how often each one appears across the full dataset.

For every flag, the notebook reports:

- the total count of rows where the flag is active
- the proportion of the dataset affected by that flag

This is a key interpretation layer. Instead of looking only at individual suspicious rows, the notebook now asks a broader question: which types of unusual behavior are common, and which are rare?

This distinction matters because a flag that appears in a handful of rows may represent a niche anomaly, while a flag that appears in a large share of the dataset may reflect either a common business pattern or an overly broad rule.

This summary helps identify which validation checks deserve more scrutiny in the next stage.

In [54]:
flag_summary = summarize_validation_flags(df_flagged)
flag_summary

,flag_name,flag_count,flag_rate
0,flag_sender_balance_mismatch,5274705,0.83
1,flag_dest_balance_mismatch,4370114,0.69
2,flag_dest_balance_zero_before_nonzero_amount,2704382,0.43
3,flag_dest_is_merchant,2151495,0.34
4,flag_org_balance_zero_before_nonzero_amount,2102433,0.33
5,flag_amount_zero,16,0.00
6,flag_amount_negative,0,0.00
7,flag_oldbalanceOrg_negative,0,0.00
8,flag_newbalanceOrig_negative,0,0.00
9,flag_oldbalanceDest_negative,0,0.00


## 16. Examining validation flags by transaction type and fraud label

This step groups the validation flags by both transaction type and fraud status.

The purpose is to move from general frequency counts to comparative pattern analysis. Instead of only asking how common a flag is overall, the notebook now asks:

- does this flag appear more often in fraud cases than non-fraud cases?
- does this flag behave differently across transaction types?
- are some suspicious patterns concentrated in particular operational contexts?

Because the flag columns are boolean values, the grouped means can be interpreted as rates. For example, a value of `0.25` means that 25 percent of rows in that group triggered the flag.

This table is very useful for identifying whether a rule-based signal is broadly distributed or concentrated in specific fraud-relevant segments.

In [55]:
flags_by_type_and_fraud = summarize_flags_by_type_and_fraud(df_flagged)
flags_by_type_and_fraud

,type,isFraud,flag_amount_negative,flag_amount_zero,flag_oldbalanceOrg_negative,flag_newbalanceOrig_negative,flag_oldbalanceDest_negative,flag_newbalanceDest_negative,flag_sender_balance_mismatch,flag_dest_balance_mismatch,flag_org_balance_zero_before_nonzero_amount,flag_dest_balance_zero_before_nonzero_amount,flag_dest_is_merchant
0,CASH_IN,0,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.01,0.12,0.00
1,CASH_OUT,0,0.00,0.00,0.00,0.00,0.00,0.00,0.92,0.29,0.46,0.14,0.00
2,CASH_OUT,1,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.20,0.01,0.30,0.00
3,DEBIT,0,0.00,0.00,0.00,0.00,0.00,0.00,0.33,0.36,0.15,0.00,0.00
4,PAYMENT,0,0.00,0.00,0.00,0.00,0.00,0.00,0.60,1.00,0.36,1.00,1.00
5,TRANSFER,0,0.00,0.00,0.00,0.00,0.00,0.00,0.97,0.29,0.53,0.12,0.00
6,TRANSFER,1,0.00,0.00,0.00,0.00,0.00,0.00,0.02,1.00,0.00,1.00,0.00


## 17. Isolating suspicious rows for closer inspection

This step creates a subset called `suspicious_rows` that includes records triggering one or more important validation flags.

The purpose is to move from summary-level patterns back to individual examples. Aggregate tables help identify broad trends, but direct row-level inspection is often necessary to understand what the flagged records actually look like in practice.

The selected conditions focus on the most important rule-based checks, especially:

- sender balance mismatches
- destination balance mismatches
- negative amounts
- negative balance values

The notebook then displays the first twenty suspicious rows. This makes it easier to manually inspect how flagged patterns appear in the raw transaction records.

In [56]:
suspicious_rows = df_flagged.loc[
    df_flagged["flag_sender_balance_mismatch"]
    | df_flagged["flag_dest_balance_mismatch"]
    | df_flagged["flag_amount_negative"]
    | df_flagged["flag_oldbalanceOrg_negative"]
    | df_flagged["flag_newbalanceOrig_negative"]
    | df_flagged["flag_oldbalanceDest_negative"]
    | df_flagged["flag_newbalanceDest_negative"]
].copy()

suspicious_rows.head(20)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,flag_amount_negative,flag_amount_zero,flag_oldbalanceOrg_negative,flag_newbalanceOrig_negative,flag_oldbalanceDest_negative,flag_newbalanceDest_negative,sender_balance_error,flag_sender_balance_mismatch,dest_balance_error,flag_dest_balance_mismatch,flag_org_balance_zero_before_nonzero_amount,flag_dest_balance_zero_before_nonzero_amount,flag_dest_is_merchant
0,1,PAYMENT,"9,839.64",C1231006815,"170,136.00","160,296.36",M1979787155,0.00,0.00,0,0,False,False,False,False,False,False,0.00,False,"9,839.64",True,False,True,True
1,1,PAYMENT,"1,864.28",C1666544295,"21,249.00","19,384.72",M2044282225,0.00,0.00,0,0,False,False,False,False,False,False,0.00,False,"1,864.28",True,False,True,True
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0,False,False,False,False,False,False,0.00,False,181.00,True,False,True,False
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,0,False,False,False,False,False,False,0.00,False,"21,363.00",True,False,False,False
4,1,PAYMENT,"11,668.14",C2048537720,"41,554.00","29,885.86",M1230701703,0.00,0.00,0,0,False,False,False,False,False,False,0.00,False,"11,668.14",True,False,True,True
5,1,PAYMENT,"7,817.71",C90045638,"53,860.00","46,042.29",M573487274,0.00,0.00,0,0,False,False,False,False,False,False,0.00,False,"7,817.71",True,False,True,True
6,1,PAYMENT,"7,107.77",C154988899,"183,195.00","176,087.23",M408069119,0.00,0.00,0,0,False,False,False,False,False,False,0.00,False,"7,107.77",True,False,True,True
7,1,PAYMENT,"7,861.64",C1912850431,"176,087.23","168,225.59",M633326333,0.00,0.00,0,0,False,False,False,False,False,False,0.00,False,"7,861.64",True,False,True,True
8,1,PAYMENT,"4,024.36",C1265012928,"2,671.00",0.00,M1176932104,0.00,0.00,0,0,False,False,False,False,False,False,"1,353.36",True,"4,024.36",True,False,True,True
9,1,DEBIT,"5,337.77",C712410124,"41,720.00","36,382.23",C195600860,"41,898.00","40,348.79",0,0,False,False,False,False,False,False,0.00,False,"6,886.98",True,False,False,False


## 18. Focusing on suspicious rows that are actual fraud cases

This step narrows the suspicious-row subset further by keeping only those rows where `isFraud == 1`.

The goal here is to examine whether flagged anomalies overlap with confirmed fraud events. This is a very useful analytical bridge between rule-based screening and label-based fraud analysis.

If suspicious validation patterns repeatedly appear in confirmed fraud rows, they may become valuable candidates for later feature engineering or anomaly-scoring logic. If they appear just as often in non-fraud rows, then they may reflect ordinary transaction mechanics rather than meaningful fraud signals.

Displaying a sample of suspicious fraud rows helps ground the analysis in actual records rather than only abstract percentages.

In [57]:
suspicious_fraud_rows = suspicious_rows.loc[suspicious_rows["isFraud"] == 1].copy()
suspicious_fraud_rows.head(20)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,flag_amount_negative,flag_amount_zero,flag_oldbalanceOrg_negative,flag_newbalanceOrig_negative,flag_oldbalanceDest_negative,flag_newbalanceDest_negative,sender_balance_error,flag_sender_balance_mismatch,dest_balance_error,flag_dest_balance_mismatch,flag_org_balance_zero_before_nonzero_amount,flag_dest_balance_zero_before_nonzero_amount,flag_dest_is_merchant
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0,False,False,False,False,False,False,0.00,False,181.00,True,False,True,False
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,0,False,False,False,False,False,False,0.00,False,"21,363.00",True,False,False,False
251,1,TRANSFER,"2,806.00",C1420196421,"2,806.00",0.00,C972765878,0.00,0.00,1,0,False,False,False,False,False,False,0.00,False,"2,806.00",True,False,True,False
252,1,CASH_OUT,"2,806.00",C2101527076,"2,806.00",0.00,C1007251739,"26,202.00",0.00,1,0,False,False,False,False,False,False,0.00,False,"29,008.00",True,False,False,False
680,1,TRANSFER,"20,128.00",C137533655,"20,128.00",0.00,C1848415041,0.00,0.00,1,0,False,False,False,False,False,False,0.00,False,"20,128.00",True,False,True,False
681,1,CASH_OUT,"20,128.00",C1118430673,"20,128.00",0.00,C339924917,"6,268.00","12,145.85",1,0,False,False,False,False,False,False,0.00,False,"14,250.15",True,False,False,False
724,1,CASH_OUT,"416,001.34",C749981943,0.00,0.00,C667346055,102.00,"9,291,620.00",1,0,False,False,False,False,False,False,"416,001.34",True,"8,875,517.00",True,True,False,False
969,1,TRANSFER,"1,277,212.75",C1334405552,"1,277,212.75",0.00,C431687661,0.00,0.00,1,0,False,False,False,False,False,False,0.00,False,"1,277,212.75",True,False,True,False
970,1,CASH_OUT,"1,277,212.75",C467632528,"1,277,212.75",0.00,C716083600,0.00,"2,444,985.25",1,0,False,False,False,False,False,False,0.00,False,"1,167,772.50",True,False,True,False
1115,1,TRANSFER,"35,063.63",C1364127192,"35,063.63",0.00,C1136419747,0.00,0.00,1,0,False,False,False,False,False,False,0.00,False,"35,063.63",True,False,True,False


## 19. Running a complete chunked audit pipeline across the full dataset

This step executes the full chunked audit pipeline in one pass.

Although the full dataset was already loaded into memory earlier, this function provides a more scalable and production-friendly version of the audit. It processes the dataset in chunks and produces a compact set of structured outputs, including:

- an audit summary
- missing-value counts
- duplicate-row count
- validation-flag summary
- class-balance summary
- transaction-type risk summary

This step is important for two reasons.

First, it confirms that the audit logic can run in a memory-aware way even if full-memory loading is not always practical.

Second, it creates a reusable audit-output object that can be saved and referenced later, which improves project reproducibility and workflow discipline.

The first displayed result is the high-level audit summary, which acts as a compact dashboard for the full quality-audit stage.

In [58]:
audit_outputs = run_chunked_quality_audit(DATA_PATH, chunksize=250_000)

audit_outputs["audit_summary"]

,metric,value
0,row_count,"6,362,620.00"
1,chunk_count,26.00
2,duplicate_rows,0.00
3,fraud_count,"8,213.00"
4,fraud_rate,0.00


## 20. Reviewing the full-dataset missing-value output from the audit pipeline

This step displays the missing-value results produced by the chunked audit pipeline.

Even though missing values were already checked earlier, showing the chunked-audit output separately is valuable because it confirms that the consolidated audit process is producing consistent results.

This is an example of analytical redundancy used in a good way. Reconfirming an important structural finding through a second, scalable pipeline increases confidence in the result.

In [59]:
audit_outputs["missing_values"]

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

## 21. Reviewing the validation-flag summary from the audit pipeline

This step displays the final validation-flag summary produced by the chunked audit pipeline.

This output is especially useful because it condenses the rule-based audit into a compact overview that can later be exported, discussed, and compared across project phases.

In practical terms, this table answers questions such as:

- which validation rules are triggered most often?
- which rules are effectively inactive?
- which suspicious patterns affect a meaningful share of the dataset?

This table will likely become one of the most important bridge artifacts between data-quality auditing and the next stage of fraud-focused exploratory analysis.

In [60]:
audit_outputs["flag_summary"]

,flag_name,flag_count,flag_rate
0,flag_sender_balance_mismatch,5274705,0.83
1,flag_dest_balance_mismatch,4370114,0.69
2,flag_dest_balance_zero_before_nonzero_amount,2704382,0.43
3,flag_dest_is_merchant,2151495,0.34
4,flag_org_balance_zero_before_nonzero_amount,2102433,0.33
5,flag_amount_zero,16,0.00
6,flag_amount_negative,0,0.00
7,flag_newbalanceDest_negative,0,0.00
8,flag_newbalanceOrig_negative,0,0.00
9,flag_oldbalanceDest_negative,0,0.00


## 22. Reviewing the class-balance output from the audit pipeline

This step displays the class-balance summary produced by the chunked audit pipeline.

Although class balance was already reviewed using the in-memory DataFrame, this second version confirms that the chunked pipeline is producing the same fraud distribution at full scale.

This consistency matters because later stages of the project may rely more heavily on chunked or scalable workflows. Confirming that the results match across methods strengthens trust in the notebook's core findings.

In [61]:
audit_outputs["class_balance"]

,isFraud,count,rate
0,0,6354407,1.00
1,1,8213,0.00


## 23. Reviewing the transaction-type risk summary from the audit pipeline

This step displays the transaction-type risk summary produced by the chunked audit pipeline.

The goal is to confirm the fraud concentration pattern found earlier and ensure that the scalable audit pipeline preserves the same relationship between transaction type and fraud exposure.

This output is one of the most business-relevant parts of the notebook because it begins to identify where fraud risk is concentrated in the system. It also helps frame the next phase of work, where deeper transaction-type analysis and temporal exploration will likely play a central role.

In [62]:
audit_outputs["type_risk_summary"]

,type,txn_count,fraud_count,amount_sum,fraud_rate,avg_amount
4,TRANSFER,532909,4097,"485,292,015,616.00",0.01,"910,647.06"
1,CASH_OUT,2237500,4116,"394,412,982,272.00",0.00,"176,273.96"
0,CASH_IN,1399284,0,"236,367,380,480.00",0.00,"168,920.23"
2,DEBIT,41432,0,"227,199,200.00",0.00,"5,483.66"
3,PAYMENT,2151495,0,"28,093,372,416.00",0.00,"13,057.61"


## 24. Saving all audit outputs for later project phases

This final step saves the structured audit outputs to the designated output directory.

Saving results is an important project-management practice. It allows later notebooks and analysis phases to reuse the audit findings without rerunning every heavy computation from scratch. It also creates a documented record of what the Phase 2 audit produced.

The saved outputs can support several future tasks, including:

- writing project summaries
- building cleaner presentation tables
- feeding later feature-engineering work
- comparing results across revised audit logic
- including evidence in the final portfolio presentation

This cell closes the notebook by turning the audit from a temporary interactive session into a reusable project artifact.

In [63]:
save_audit_outputs(audit_outputs, OUTPUT_DIR)
print(f"Saved audit outputs to: {OUTPUT_DIR.resolve()}")

Saved audit outputs to: /Users/twon/Documents/Time Series Analysis Project/data/outputs/phase_2_audit


## Conclusion

This notebook completed the full data-quality audit for the fraud detection dataset.

At the end of this phase, the project now has a verified understanding of the dataset at full scale, including:

- full dataset size
- structural schema and memory profile
- missing-value status
- duplicate-row status
- numeric range profile
- fraud class imbalance
- fraud concentration by transaction type
- rule-based validation flags for suspicious patterns
- saved audit outputs for downstream use

The most important outcome of this phase is not only that the dataset has been checked, but that its strengths and limitations are now clearer.

The dataset appears structurally clean in terms of missing values and exact duplicates. At the same time, the fraud label is extremely imbalanced, and the rule-based balance checks suggest that some transaction mechanics may be more complex than a simple one-rule interpretation would imply.

This means the project is now ready to move into the next analytical stage with more confidence. The next phase should focus on deeper exploratory analysis, especially around transaction-type behavior, temporal fraud patterns, and the interpretation of suspicious balance-related signals.